# Purdue University Affiliation Analysis

This notebook analyzes awardee names for Purdue University connections using the Institution Checker pipeline.

**Workflow:**
1. **Setup**: Configure environment, reload modules, and set API keys.
2. **Data Loading**: Load and prepare the dataset (Full or Sample).
3. **Clustering**: Group similar names to optimize processing.
4. **Validation**: Run a quick test on known VIPs to ensure accuracy.
5. **Execution**: Run the full pipeline on the clustered dataset.
6. **Analysis**: Review results, verify specific targets, and export data.

In [ ]:
# 1. SETUP & CONFIGURATION
# -----------------------------------------------------------------------------
# This cell handles module reloading and API configuration.
# Run this FIRST to ensure the environment is ready.

import sys
import importlib
import pandas as pd
import time

# --- Module Reloading (Development Mode) ---
# Reinstall local package to ensure latest changes are applied
!pip uninstall -y institution_checker
!pip install -e .

# Force reload of modules to pick up changes
modules_to_reload = [
    'institution_checker.config',
    'institution_checker.search',
    'institution_checker.llm_processor',
    'institution_checker.main',
    'institution_checker'
]

for module_name in modules_to_reload:
    if module_name in sys.modules:
        importlib.reload(sys.modules[module_name])
        print(f"✅ Reloaded {module_name}")

# --- Imports (AFTER Reloading) ---
# Import AFTER reloading to ensure we get the new function objects
import institution_checker
import institution_checker.llm_processor
import institution_checker.search
from institution_checker import run_pipeline, resolve_names, set_api_key
from institution_checker.llm_processor import enable_prompt_logging
from name_matcher.pipeline import NameMatcher, NameMatcherConfig

# --- API Configuration ---
API_KEY = "sk-3f1dbbf2450e46ab9541dffba4f18ec6"
set_api_key(API_KEY)

# Configure NameMatcher
config = NameMatcherConfig(name_column="name", use_tqdm=True)
config.llm_token = API_KEY

print("\n✅ Setup complete:")
print("   - Modules reloaded and installed")
print("   - API keys configured")
print("   - Browser timeout fixes active (15s)")
print("   - Search strategies optimized (DuckDuckGo prioritized + Rate Limit Protection)")
print("   - Evidence gathering expanded (8 excerpts)")

Found existing installation: institution-checker 0.1.0
Uninstalling institution-checker-0.1.0:
  Successfully uninstalled institution-checker-0.1.0
Obtaining file:///C:/Users/setiawa/Documents/institution_checker
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Checking if build backend supports build_editable: started
  Checking if build backend supports build_editable: finished with status 'done'
  Getting requirements to build editable: started
  Getting requirements to build editable: finished with status 'done'
  Preparing editable metadata (pyproject.toml): started
  Preparing editable metadata (pyproject.toml): finished with status 'done'
  Using cached h11-0.14.0-py3-none-any.whl.metadata (8.2 kB)
  Using cached urllib3-1.26.20-py2.py3-none-any.whl.metadata (50 kB)
  Using cached websockets-10.4-cp312-cp312-win_amd64.whl
Using cached h11-0.14.0-py3-none-any.whl (58 kB)
Using cached urllib3-1.26.20-py2.py3-none-any.whl (144 

  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
conda-repo-cli 1.0.114 requires urllib3>=2.2.2, but you have urllib3 1.26.20 which is incompatible.
crossrefapi 1.6.0 requires urllib3==1.26.16, but you have urllib3 1.26.20 which is incompatible.
habanero 2.2.0 requires urllib3==2.2.0, but you have urllib3 1.26.20 which is incompatible.
seleniumbase 4.45.10 requires attrs>=25.4.0, but you have attrs 25.3.0 which is incompatible.
seleniumbase 4.45.10 requires certifi>=2026.1.4, but you have certifi 2025.8.3 which is incompatible.
seleniumbase 4.45.10 requires h11==0.16.0, but you have h11 0.14.0 which is incompatible.
seleniumbase 4.45.10 requires selenium==4.39.0; python_version >= "3.10", but you have selenium 4.23.1 which is incompatible.
seleniumbase 4.45.10 requires trio<1,>=0.32.0; python_version >= "3.10", but you have t


✅ Setup complete:
   - Modules reloaded and installed
   - API keys configured
   - Browser timeout fixes active (15s)
   - Search strategies optimized (DuckDuckGo prioritized + Rate Limit Protection)
   - Evidence gathering expanded (8 excerpts)


In [2]:
# 2. DATA LOADING & PREPROCESSING
# -----------------------------------------------------------------------------
# Select the dataset to process and prepare it for analysis.

# --- Configuration ---
DATASET_OPTION = "purdue"  # Options: "purdue", "nobel", "sample", "accuracy_test_25"

# File Paths
PATHS = {
    "purdue": r"E:\Ace\To be sorted\Purdue Trial Set HP Awardees 2024-10-23.xlsx",
    "nobel": r"E:\Ace\Nobel Only Test Set.xlsx"
}

# --- Load Data ---
if DATASET_OPTION == "sample":
    print("📉 CREATING TEST SAMPLE: 9 VIPs + 100 Random Names")
    # Load base dataset (Nobel) for sampling
    df_base = pd.read_excel(PATHS["nobel"], dtype=str)
    
    vip_names = [
        "Akira Suzuki", "Ei-ichi Negishi", "Herbert C. Brown", 
        "John B. Fenn", "Vernon L. Smith", "Ben R. Mottelson", 
        "E. M. Purcell", "Julian Schwinger", "Wolfgang Pauli"
    ]
    
    vip_mask = df_base['name'].apply(lambda x: any(vip.lower() in str(x).lower() for vip in vip_names))
    df_vips = df_base[vip_mask].copy()
    df_others = df_base[~vip_mask].copy()
    
    sample_size = min(100, len(df_others))
    df_random = df_others.sample(n=sample_size, random_state=42)
    
    df = pd.concat([df_vips, df_random]).drop_duplicates(subset=['name']).reset_index(drop=True)
    print(f"✅ Sample created: {len(df)} records")

elif DATASET_OPTION == "accuracy_test_25":
    print("📉 CREATING ACCURACY TEST: 9 VIPs + 16 Random Non-Connected")
    df_base = pd.read_excel(PATHS["nobel"], dtype=str)
    
    vip_names = [
        "Akira Suzuki", "Ei-ichi Negishi", "Herbert C. Brown", 
        "John B. Fenn", "Vernon L. Smith", "Ben R. Mottelson", 
        "E. M. Purcell", "Julian Schwinger", "Wolfgang Pauli"
    ]
    
    # 1. Get VIPs
    vip_mask = df_base['name'].apply(lambda x: any(vip.lower() in str(x).lower() for vip in vip_names))
    df_vips = df_base[vip_mask].drop_duplicates(subset=['name']).copy()
    
    # 2. Get Others (Non-VIPs)
    df_others = df_base[~vip_mask].copy()
    
    # 3. Select 16 random names
    # Using a fixed seed ensures reproducibility. 
    df_random = df_others.sample(n=16, random_state=123) 
    
    # Combine
    df = pd.concat([df_vips, df_random]).drop_duplicates(subset=['name']).reset_index(drop=True)
    
    print(f"✅ Test set created: {len(df)} records")
    print(f"   VIPs (Positive Controls): {len(df_vips)}")
    print(f"   Others (Negative Controls): {len(df_random)}")
    
    print("\n📋 List of names to check:")
    print("-" * 30)
    for i, name in enumerate(df['name'], 1):
        role = "VIP" if any(v in name for v in vip_names) else "Other"
        print(f"{i:2}. {name:<30} [{role}]")

elif DATASET_OPTION in PATHS:
    path = PATHS[DATASET_OPTION]
    df = pd.read_excel(path, dtype=str)
    print(f"✅ Loaded {len(df):,} records from {DATASET_OPTION.upper()} dataset")
    print(f"   Source: {path}")

else:
    raise ValueError(f"Invalid DATASET_OPTION: {DATASET_OPTION}")

# --- Clustering ---
print("\n🧩 Clustering similar names...")
matcher = NameMatcher(config)
result = matcher.cluster(df)
df_clustered = result.dataframe
print(f"✅ Clustering complete: {len(df)} names -> {df_clustered['cluster_label'].nunique()} clusters")


✅ Loaded 607 records from PURDUE dataset
   Source: E:\Ace\To be sorted\Purdue Trial Set HP Awardees 2024-10-23.xlsx

🧩 Clustering similar names...
--- Name Matcher Process Started (v6.8.8 Improved Canonical Name Selection) ---

1. Loading and preparing data...
   Loaded 607 names. Done in 0.02s
2. Vectorizing names for similarity scoring...
   Done in 0.02s
3. Generating candidate pairs via Blocking...
   Generated 682 candidate pairs from blocking.
   Done in 0.00s
4. Applying Stricter Cluster-Centric Filtering...


   Filtering Pairs: 100%|██████████| 682/682 [00:00<00:00, 341391.02pair/s]


   Filter Stats: {'merged_expansion': 299, 'rejected_transitive_conflict': 45}
   Done in 0.00s
5. Reviewing ambiguous pairs with LLM...
   Sending 6 unique pairs to LLM for review using 5 parallel workers...


   LLM Review: 100%|██████████| 1/1 [00:04<00:00,  4.48s/batch]

   Done in 4.48s
6. Finalizing cluster data structures...
   Done in 0.00s
7. Refining clusters by ejecting outliers...
   Found and ejected 0 outliers from clusters.
   Done in 0.07s
8. Assigning canonical names and saving results...

--- Results Summary ---
   - Total names processed: 607
   - Unique entities (clusters) found: 305

   --- Sample of Largest Clusters Found ---
   Cluster 1 (Size: 10): 'Seymour Benzer'
     - Seymour Benzer
     - Seymour Benzer
     - Seymour Benzer
     - Seymour Benzer
     - Seymour Benzer
     - ...
   Cluster 2 (Size: 8): 'Alvin Carl Plantinga'
     - Alvin C. Plantinga
     - Alvin C. Plantinga
     - Alvin C. Plantinga
     - Alvin C. Plantinga
     - Alvin C. Plantinga
     - ...
   Cluster 3 (Size: 7): 'George Andrew Olah'
     - George A. Olah
     - George A. Olah
     - George A. Olah
     - George A. Olah
     - George A. Olah
     - ...
   Cluster 4 (Size: 7): 'Hans Albrecht Bethe'
     - H. A. Bethe
     - H. A. Bethe
     - Hans A. Beth

In [3]:
# 3. PIPELINE VALIDATION (VIP TEST)
# -----------------------------------------------------------------------------
# Run a quick test on known VIPs to verify pipeline accuracy before the full run.

vip_test_names = [
    "Akira Suzuki", "Ei-ichi Negishi", "Herbert C. Brown", 
    "John B. Fenn", "Vernon L. Smith", "Ben R. Mottelson", 
    "E. M. Purcell", "Julian Schwinger", "Wolfgang Pauli"
]

print("🧪 VIP ACCURACY TEST (9 names)")
print("=" * 60)
print("Testing pipeline accuracy...")

start_time = time.time()

# Run pipeline on VIPs
vip_results = await run_pipeline(
    vip_test_names, 
    batch_size=3,  # Reduced to avoid DDG rate limits (was 9)
    use_enhanced_search=True,
    dataset_profile="high_connection",
    use_dynamic_batching=False
)

elapsed = time.time() - start_time

# Analyze Results
print(f"\n📊 VIP TEST RESULTS ({elapsed:.1f}s, {elapsed/9:.1f}s per name):")
print("-" * 60)

vip_connected = 0
for name, res in zip(vip_test_names, vip_results):
    verdict = res.get('verdict', 'unknown')
    institution = res.get('institution', '')
    is_purdue = 'purdue' in institution.lower() if institution else False
    is_connected = verdict == 'connected' and is_purdue
    
    if is_connected:
        vip_connected += 1
        icon = "✅"
    else:
        icon = "❌"
    
    print(f"   {icon} {name}: {verdict}")

recall = (vip_connected / 9) * 100
print(f"\n🏆 VIP RECALL: {vip_connected}/9 ({recall:.0f}%)")

if vip_connected >= 8:
    print("✅ Accuracy OK - proceed with full dataset")
else:
    print("⚠️  Low accuracy - check module reload (Cell 1) before proceeding")

🧪 VIP ACCURACY TEST (9 names)
Testing pipeline accuracy...
[PIPELINE] Starting: 9 name(s) in 3 batch(es) using enhanced search
[PIPELINE] Batch size: 3, Inter-batch delay: 0.5s


Processing batches:   0%|          | 0/3 [00:00<?, ?batch/s]


[PIPELINE] ===== BATCH 1/3 =====
[PIPELINE] Names in this batch: ['Akira Suzuki', 'Ei-ichi Negishi', 'Herbert C. Brown']
[BATCH] Processing 3 names: Akira Suzuki, Ei-ichi Negishi, Herbert C. Brown
[BATCH] Phase 1: Running searches in parallel for all 3 names (max 8 concurrent)
[PROGRESS] Starting search for: Ei-ichi Negishi
[PROGRESS] Name: 'Ei-ichi Negishi' | Query: 'Ei-ichi Negishi Purdue University'
[PROGRESS] Starting search for: Akira Suzuki
[PROGRESS] Name: 'Akira Suzuki' | Query: 'Akira Suzuki Purdue University'
[PROGRESS] Starting search for: Herbert C. Brown
[PROGRESS] Name: 'Herbert C. Brown' | Query: 'Herbert C. Brown Purdue University'
[PROGRESS] Result sample: Herbert Charles Brown | Biography, Significance, Discoveries ... Herbert C. Brow
[PROGRESS] Basic search returned 19 results, low quality (max_score=20), escalating to enhanced search...
[PROGRESS] Result sample: Ei-ichi Negishi, Alumnus and Nobel Laureate | University of ...
[PROGRESS] Basic search returned 20 resu

In [4]:
# 4. MAIN EXECUTION (FULL DATASET)
# -----------------------------------------------------------------------------
# Run the affiliation checker on the full clustered dataset using the optimized library pipeline.

# --- Configuration ---
TEST_MODE = False        # Set True to check only first 20 clusters
DATASET_PROFILE = "low_connection"  # "low_connection" for strict filtering (Nobel)
BATCH_SIZE = 20         # Search batch size (LLM will auto-flush at 5)

# --- Prepare Data ---
# Get cluster representatives
cluster_reps = df_clustered.groupby('cluster_label')['name'].first().reset_index()
cluster_reps.columns = ['cluster_label', 'representative_name']

if TEST_MODE:
    cluster_reps = cluster_reps.head(20)
    print(f"🧪 TEST MODE: Checking {len(cluster_reps)} clusters")
else:
    print(f"🚀 Checking all {len(cluster_reps)} clusters")

names_to_check = cluster_reps['representative_name'].tolist()

# --- Run Pipeline ---
print(f"🔍 Starting Optimized Institution Algorithm (internal library v2)...")
print(f"   Profile: {DATASET_PROFILE} (Aggressive Filtering)")
print(f"   Search Batch: {BATCH_SIZE}")
print(f"   Names to process: {len(names_to_check)}")
print("-" * 60)

start_time = time.time()

# We rely on the internal dynamic batching which now implements the
# Search Batch (20) -> Queue -> LLM Flush (5) architecture.
cluster_results = await run_pipeline(
    names_to_check, 
    batch_size=BATCH_SIZE,
    use_enhanced_search=True,
    dataset_profile=DATASET_PROFILE,
    use_dynamic_batching=True
)

elapsed = time.time() - start_time
avg_per_name = elapsed / len(names_to_check) if names_to_check else 0

print(f"\n⏱️ TIMING:")
print(f"   Total: {elapsed:.1f}s")
print(f"   Average: {avg_per_name:.2f}s per name")
print(f"   Rate: {3600/avg_per_name:.0f} names/hour")

# --- Map Results ---
cluster_institution_map = {}
for cluster_label, rep_name, res in zip(cluster_reps['cluster_label'], cluster_reps['representative_name'], cluster_results):
    if not isinstance(res, dict):
        res = {'raw_result': res}
    
    institution = res.get('institution', '')
    verdict = res.get('verdict', '')
    purdue_match = (verdict == 'connected' and 
                   ('purdue university' in str(institution).lower() or 
                   (str(institution).lower().startswith('purdue') and 'university' in str(institution).lower())))
    
    res['purdue_connection'] = purdue_match
    cluster_institution_map[cluster_label] = res

fields_to_map = ['institution', 'verdict', 'relationship_type', 'relationship_timeframe', 
                 'verification_detail', 'summary', 'primary_source', 'verification_status', 
                 'temporal_context', 'confidence', 'purdue_connection']

for field in fields_to_map:
    df_clustered[field] = df_clustered['cluster_label'].map(
        lambda x: cluster_institution_map.get(x, {}).get(field, None)
    )

final_results = df_clustered
print("✅ Affiliation check complete")

🚀 Checking all 305 clusters
🔍 Starting Optimized Institution Algorithm (internal library v2)...
   Profile: low_connection (Aggressive Filtering)
   Search Batch: 20
   Names to process: 305
------------------------------------------------------------
[PIPELINE] Starting DYNAMIC BATCHING mode
[PIPELINE] Total: 305 names, Search batch: 20, LLM Flush Size: 5
[PIPELINE] Using enhanced search, Dataset profile: low_connection


Processing search batches:   0%|          | 0/16 [00:00<?, ?batch/s]


[PIPELINE] ===== SEARCH BATCH 1/16 =====
[PIPELINE] Names: A. Leon Higginbotham, A. Stephen Morse, Abraham Tesser, Akasha Gloria Hull, Akira Suzuki, Alan Jay Perlis, Albert W. Overhauser, Alexandra Boltasseva, Alfred E. Mann, Alice H. Eagly, Alondra Nelson, Alvin C. Plantinga, Amit Goyal, Andrew H. Bobeck, Andrew J. Majda, Andrew M. Weiner, Andrew P. Sage, Andrey A. Potter, Arden L. Bement, A. Dale Kaiser
[SEARCH] Running parallel searches for 20 names (max 8 concurrent)...
[PROGRESS] Starting search for: Albert W. Overhauser
[PROGRESS] Name: 'Albert W. Overhauser' | Query: 'Albert W. Overhauser Purdue University'
[PROGRESS] Starting search for: Andrew H. Bobeck
[PROGRESS] Name: 'Andrew H. Bobeck' | Query: 'Andrew H. Bobeck Purdue University'
[PROGRESS] Starting search for: Akasha Gloria Hull
[PROGRESS] Name: 'Akasha Gloria Hull' | Query: 'Akasha Gloria Hull Purdue University'
[PROGRESS] Starting search for: Alan Jay Perlis
[PROGRESS] Name: 'Alan Jay Perlis' | Query: 'Alan Jay Perlis 

In [5]:
# 5. ANALYSIS & EXPORT
# -----------------------------------------------------------------------------
# Analyze results, verify VIPs in the full dataset, and export to Excel.

# --- Configuration ---
EXPORT_FILENAME = 'institution_check_results.xlsx'
VIP_TARGETS = [
    "Akira Suzuki", "Ei-ichi Negishi", "Herbert C. Brown", 
    "John B. Fenn", "Vernon L. Smith", "Ben R. Mottelson", 
    "E. M. Purcell", "Julian Schwinger", "Wolfgang Pauli"
]

print("📊 RESULTS ANALYSIS")
print("=" * 70)

# 1. General Statistics
if 'final_results' in locals():
    total_records = len(final_results)
    purdue_records = final_results[final_results['purdue_connection'] == True]
    percentage = (len(purdue_records) / total_records * 100) if total_records > 0 else 0

    print(f"   Total records: {total_records:,}")
    print(f"   Purdue-affiliated: {len(purdue_records):,} ({percentage:.1f}%)")

    # 2. VIP Verification (Recall Check)
    print(f"\n🏆 VIP RECALL CHECK (Known Purdue Connections)")
    print("-" * 70)
    vip_hits = 0
    for target in VIP_TARGETS:
        # Find in results
        match = final_results[final_results['name'].str.contains(target, case=False, na=False)]
        if not match.empty:
            row = match.iloc[0]
            is_connected = row.get('purdue_connection', False)
            verdict = row.get('verdict', 'N/A')
            rel_type = row.get('relationship_type', 'N/A')
            icon = "✅" if is_connected else "❌"
            print(f"   {icon} {target:<20} : {verdict} ({rel_type})")
            if is_connected: vip_hits += 1
        else:
            print(f"   ❓ {target:<20} : Not found in dataset")

    print(f"\n   Recall Score: {vip_hits}/{len(VIP_TARGETS)} ({(vip_hits/len(VIP_TARGETS)*100):.0f}%)")

    # 3. False Positive Check (Non-VIPs)
    print(f"\n🚨 POTENTIAL FALSE POSITIVES (Non-VIPs marked Connected)")
    print("-" * 70)
    non_vip_connected = []
    for idx, row in final_results.iterrows():
        name = row['name']
        is_vip = any(vip.lower() in str(name).lower() for vip in VIP_TARGETS)
        is_connected = row.get('purdue_connection', False)
        
        if is_connected and not is_vip:
            non_vip_connected.append(row)
            print(f"   ⚠️ {name:<20} : {row.get('relationship_type', 'N/A')}")
            print(f"      Detail: {str(row.get('verification_detail', ''))[:100]}...")

    if not non_vip_connected:
        print("   ✅ No False Positives detected (among non-VIPs)")
    else:
        print(f"   ⚠️ {len(non_vip_connected)} False Positives detected")

    # 4. Export
    print(f"\n💾 EXPORTING RESULTS")
    print("-" * 70)
    try:
        final_results.to_excel(EXPORT_FILENAME, index=False)
        print(f"   ✅ Saved to: {EXPORT_FILENAME}")
        
        # Save Purdue-only subset
        purdue_only = final_results[final_results['purdue_connection'] == True]
        purdue_filename = EXPORT_FILENAME.replace('.xlsx', '_purdue_only.xlsx')
        purdue_only.to_excel(purdue_filename, index=False)
        print(f"   ✅ Saved to: {purdue_filename}")
    except Exception as e:
        print(f"   ❌ Export failed: {e}")
else:
    print("⚠️ 'final_results' dataframe not found. Run Section 4 first.")

print("\n" + "=" * 70)


📊 RESULTS ANALYSIS
   Total records: 607
   Purdue-affiliated: 379 (62.4%)

🏆 VIP RECALL CHECK (Known Purdue Connections)
----------------------------------------------------------------------
   ✅ Akira Suzuki         : connected (Postdoc)
   ✅ Ei-ichi Negishi      : connected (Faculty)
   ❌ Herbert C. Brown     : not_connected (None)
   ✅ John B. Fenn         : connected (Attended)
   ✅ Vernon L. Smith      : connected (Faculty)
   ✅ Ben R. Mottelson     : connected (Alumni)
   ✅ E. M. Purcell        : connected (Alumni)
   ✅ Julian Schwinger     : connected (Faculty)
   ❓ Wolfgang Pauli       : Not found in dataset

   Recall Score: 7/9 (78%)

🚨 POTENTIAL FALSE POSITIVES (Non-VIPs marked Connected)
----------------------------------------------------------------------
   ⚠️ A. Dale Kaiser       : Alumni
      Detail: “B.S., Purdue University, Science (1950)” – indicates Kaiser earned a Bachelor of Science degree fro...
   ⚠️ A. Dale Kaiser       : Alumni
      Detail: “B.S., Purdue 

In [6]:
# 6. DEBUGGING TOOLS (OPTIONAL)
# -----------------------------------------------------------------------------
# Use this section to investigate specific names that are failing or giving unexpected results.

from institution_checker.search import enhanced_search
from institution_checker.main import should_skip_llm, _aggregate_result_text

# --- Configuration ---
DEBUG_NAMES = ["Akira Suzuki"]  # Add names to debug here

for name in DEBUG_NAMES:
    print(f"\n{'='*70}")
    print(f"🔍 DEBUGGING: {name}")
    print('='*70)
    
    # 1. Run Search
    print(f"\n1️⃣ Running enhanced search...")
    results = await enhanced_search(name, "Purdue University", num_results=20, debug=True)
    print(f"   Found {len(results)} results")
    
    # 2. Analyze Top Results
    print(f"\n2️⃣ Top search results:")
    for i, r in enumerate(results[:5], 1):
        score = r.get('signals', {}).get('relevance_score', 0)
        title = r.get('title', 'N/A')[:60]
        url = r.get('url', '')[:50]
        snippet = r.get('snippet', '')[:100]
        has_inst = r.get('signals', {}).get('has_institution', False)
        print(f"   {i}. [Score: {score:2}] {title}")
        print(f"      URL: {url}...")
        print(f"      Snippet: {snippet}...")
        print(f"      Signals: institution={has_inst}")
    
    # 3. Check Skip Logic
    print(f"\n3️⃣ Skip logic evaluation:")
    should_skip, reason = should_skip_llm(results, dataset_profile="high_connection")
    print(f"   Should skip: {should_skip}")
    print(f"   Reason: {reason}")
    
    # 4. Run Full Pipeline (Single Item)
    print(f"\n4️⃣ Running full pipeline...")
    test_results = await run_pipeline(
        [name], 
        batch_size=1, 
        use_enhanced_search=True, 
        dataset_profile="high_connection", 
        use_dynamic_batching=False
    )
    
    res = test_results[0]
    print(f"   Verdict: {res.get('verdict')}")
    print(f"   Institution: {res.get('institution')}")
    print(f"   Relationship: {res.get('relationship_type')}")
    print(f"   Detail: {res.get('verification_detail', '')[:200]}...")


🔍 DEBUGGING: Akira Suzuki

1️⃣ Running enhanced search...
[search] Early exit threshold: 5 (profile: default)
[search] Attempting DuckDuckGo first for speed/reliability...
[search] Using duckduckgo_search library for: Akira Suzuki Purdue University
[search] Attempting DuckDuckGo first for speed/reliability...
[search] Using duckduckgo_search library for: "Akira Suzuki" site:purdue.edu
[search] Attempting DuckDuckGo first for speed/reliability...
[search] Using duckduckgo_search library for: "Akira Suzuki" Purdue University (professor OR faculty OR alumni OR resume OR CV OR visiting OR adjunct)
[search] Attempting DuckDuckGo first for speed/reliability...
[search] Using duckduckgo_search library for: "Akira Suzuki" Purdue University (degree OR B.S. OR B.A. OR PhD OR student OR graduated OR 19*)
[search] duckduckgo_search library returned 5 raw results
[search] duckduckgo_search library yielded 5 valid results
[search] DuckDuckGo returned 5 results, skipping Bing
[search] duckduckgo_sea

Processing batches:   0%|          | 0/1 [00:00<?, ?batch/s]


[PIPELINE] ===== BATCH 1/1 =====
[PIPELINE] Names in this batch: ['Akira Suzuki']
[BATCH] Processing 1 names: Akira Suzuki
[BATCH] Phase 1: Running searches in parallel for all 1 names (max 8 concurrent)
[PROGRESS] Starting search for: Akira Suzuki
[PROGRESS] Name: 'Akira Suzuki' | Query: 'Akira Suzuki Purdue University'
[PROGRESS] Result sample: Akira Suzuki - A Superstar of Science
[PROGRESS] Basic search returned 16 results, low quality (max_score=19), escalating to enhanced search...
[PROGRESS] Search completed for Akira Suzuki in 6.7s, found 20 results
[BATCH] Phase 1 completed in 7.3s
[BATCH] Search succeeded for Akira Suzuki: 20 results
[BATCH] Phase 2: Evaluating search results and running LLM analysis in parallel for all 1 names (max 10 concurrent)
[PROGRESS] Starting LLM analysis for: Akira Suzuki
[PROGRESS] LLM analysis completed for Akira Suzuki in 4.1s
[BATCH] Phase 2 completed in 4.1s (1 LLM calls, 0 skipped)
[BATCH] Completed 1/1 (Akira Suzuki) in 11.4s
[OK] Akira Suzuk